# 05.COMPLETO · Fase 1: efecto propio y efectos dentro de la categoría

Combina lo mejor de las dos versiones del notebook 05: la del grupo (`05_fase1_efecto_propio_editado`) y la del profesor guía con errores estándar corregidos. Estima, para cada categoría objetivo $j$, el modelo PPML con efectos fijos de la fase 1 de `metodologia.tex` (D‑08, D‑15, D‑16). Responde las preguntas (i) y (ii): ¿la promoción de un SKU aumenta o canibaliza las ventas del mismo SKU (el día y los días siguientes) y las de otros SKUs de su clase y de su categoría?

**Categorías.** Las 6 categorías **sin programa semanal** (BEVERAGES, GROCERY I, CLEANING, DAIRY, BREAD/BAKERY, LIQUOR,WINE,BEER): el mínimo del póster según `INSTRUCCIONES.md`. Las 4 con programa (MEATS, POULTRY, DELI, PRODUCE) quedan fuera de este notebook porque sus rezagos tienen un problema pendiente de decisión (ver `05.2`).

| | |
|---|---|
| **Entrada** | `panel_t44_sku_dia.parquet`, `exposiciones_t44_sku_dia.parquet`, `calendario_t44.csv`, `tabla_04_ventas_con_sin_promocion.csv` |
| **Salidas** | `tabla_24_fase1c_<CATEGORIA>.csv`, `tabla_24_fase1c_consolidada.csv`, `tabla_25_fase1c_bases_y_controles.csv`, `tabla_26_fase1c_efecto_propio.csv` en `resultados/tablas/` |

**Qué se tomó de cada versión.**

| De la versión del grupo | De la versión del profesor guía (`05`) |
|---|---|
| `efecto_acum_1dia`: efecto de una promoción de **un día**, con método delta (la suma β₀+β₁+β₂ es el de una promoción sostenida 7 días, no el de un día) | Errores estándar con la corrección de Cameron, Gelbach y Miller (2011) cuando la matriz de varianzas no es válida, y también guardados sin corregir |
| Control exacto de días: `== 532` (590 − 46 del terremoto − 12 por los rezagos), que detecta un día de más o de menos | Las uniones con `validate` y `assert` de filas y unicidad SKU‑día; conteo de cada exclusión |
| `n_obs` del modelo y `n_obs_descartadas` (SKUs de una sola fila que descarta `pyfixest`) | Verificación independiente de los coeficientes con otro programa (`statsmodels`) |
| Comparación con la tabla 4 en la misma escala (razón del modelo 1 + efecto/100 frente a razón de medias) | Controles de tamaño de coeficientes, tabla de controles y figura |
| Conteo de filas con nulos y en qué variables | Efecto en porcentaje con IC, y por +10 pp para las proporciones |

**Cómo se lee cada fila.** Un coeficiente $\beta$ implica que las ventas cambian en $100\,(e^{\beta}-1)\,\%$ cuando la variable aumenta en una unidad.

- `onpromotion`: efecto el día en que el SKU está en promoción.
- `onpromotion_l1_3`, `onpromotion_l4_6`: efecto de haber estado en promoción **durante los 3 días** anteriores (1 a 3 días atrás) o los 3 días siguientes a esos (4 a 6 días atrás). Son promedios de 3 días.
- `promo_clase_otros`, `promo_cat_otras_clases`: son proporciones entre 0 y 1; el coeficiente es por pasar de 0 % a 100 % del surtido en promoción. Se agrega el efecto de **10 puntos porcentuales**.
- `NETO_7dias`: $e^{\beta_0+\beta_{\ell_{1-3}}+\beta_{\ell_{4-6}}}-1$. Es el efecto de estar en promoción **7 días seguidos**, no el de una promoción de un día.
- `efecto_acum_1dia`: lo que una promoción de **un solo día** suma a las ventas de la semana del SKU, respecto de una semana normal. Es $\frac{1}{7}\left[(e^{\beta_0}-1)+3(e^{\beta_{\ell_{1-3}}/3}-1)+3(e^{\beta_{\ell_{4-6}}/3}-1)\right]$ y supone el mismo nivel base en los 7 días.

## Guía rápida: qué se hace aquí y qué significan las palabras

**La pregunta.** Cuando un producto de un supermercado está en promoción, ¿sus ventas suben? ¿cuánto? ¿y qué pasa con las ventas de los productos parecidos? Eso se mide con datos de una tienda de Quito (la tienda 44, de enero 2016 a agosto 2017).

**La idea del método, con una analogía.** Imagina que quieres saber si un alumno rinde más cuando estudia con música. Lo justo es comparar al alumno **consigo mismo**: sus notas con música frente a sus notas sin música, en el mismo tipo de día. No se compara con otro alumno, que tiene otro nivel. El modelo hace lo mismo con cada producto: compara sus ventas cuando está en promoción con sus ventas cuando no lo está, descontando el día de la semana, el mes y los festivos.

| Palabra | Qué significa, en simple |
|---|---|
| **SKU / producto** | Cada artículo distinto que vende la tienda (por ejemplo, una gaseosa de 2 litros). |
| **Categoría** | Grupo grande de productos (bebidas, lácteos, limpieza…). |
| **Clase** | Grupo chico de productos muy parecidos dentro de una categoría (por ejemplo, las gaseosas dentro de bebidas). |
| **Promoción propia** | El producto mismo está en promoción. |
| **Rezago** | Mirar días anteriores. `_l1_3` = ¿estuvo en promoción en los 3 días anteriores?; `_l4_6` = ¿entre 4 y 6 días atrás? Sirve para ver qué pasa **después** de una promoción: las ventas siguen altas, caen (los clientes se adelantaron) o vuelven a lo normal. |
| **Efectos fijos** | Le dan a cada producto, cada día de la semana y cada mes su propio nivel normal de ventas, para no confundirlos con el efecto de la promoción. |
| **Coeficiente (β)** | El número que estima el modelo para cada variable. No se lee directo: se traduce a porcentaje con 100 × (e^β − 1). |
| **Efecto en %** | Cuánto cambian las ventas. +60 % significa que venden 1,6 veces lo normal. |
| **Error estándar** | Qué tan incierto es un número. Más chico = más preciso. |
| **Valor p** | Probabilidad de ver un resultado así de grande si en realidad no hubiera efecto. Menor que 0,05 = "estadísticamente significativo": es difícil que sea pura casualidad. Mayor que 0,05 = no se distingue de cero. |
| **Intervalo de confianza (IC 95 %)** | Rango donde, con 95 % de confianza, está el valor verdadero. Si el rango incluye el cero, no se puede asegurar que haya efecto. |
| **"Asociado", no "causa"** | El modelo muestra que dos cosas van juntas, no prueba que una cause la otra: la cadena decide cuándo promocionar y puede hacerlo cuando espera vender más. |

**Un detalle que confunde: tres "efectos" distintos de la promoción propia.**
- **Efecto del día** (`onpromotion`): lo que pasa el mismo día de la promoción.
- **Promoción de 7 días seguidos** (`NETO_7dias`): lo que se asocia a estar en promoción toda una semana. Es la suma de los tres coeficientes.
- **Promoción de 1 día, efecto en la semana** (`efecto_acum_1dia`): lo que una promoción de un solo día suma a las ventas de la semana completa del producto, comparada con una semana normal. Es el más realista para una promoción de un día.

In [1]:
# ------------------------------------------------------------------------------------------------------------
# Este notebook estima la fase 1 de la metodología (efecto propio y dentro de la categoría) combinando las dos
# versiones del notebook 05, para las categorías sin programa semanal.
# ------------------------------------------------------------------------------------------------------------
# Fecha de versión: 2026-10-22
# ------------------------------------------------------------------------------------------------------------

# Librerías
# ------------------------------------------------------------------------------------------------------------
import os                                       # para manejar carpetas y rutas de archivos
import re                                       # para limpiar textos (por ejemplo, nombres de categorías)
import warnings                                 # para controlar los avisos de Python

import numpy as np                              # cálculos matemáticos (exponenciales, raíces, matrices)
import pandas as pd                             # tablas de datos (como un Excel dentro de Python)
import pyfixest as pf                           # estima el modelo estadístico (regresión Poisson con efectos fijos)
import statsmodels.api as sm                    # otro programa de estadística, solo para verificar los resultados
from scipy import stats                         # distribuciones de probabilidad (para valores p e intervalos)
import matplotlib.pyplot as plt                 # dibuja los gráficos

In [2]:
# Parámetros
# ------------------------------------------------------------------------------------------------------------
# Categorías objetivo: las 6 sin programa semanal (mínimo del póster, INSTRUCCIONES.md sección 6); la primera es la piloto
# Las categorías (tipos de producto) que vamos a analizar: las 6 que NO tienen un programa semanal fijo de promociones.
# BEVERAGES = bebidas, GROCERY I = abarrotes, CLEANING = limpieza, DAIRY = lácteos, BREAD/BAKERY = panadería,
# LIQUOR,WINE,BEER = licores y cervezas.
categorias_objetivo = ["BEVERAGES", "GROCERY I", "CLEANING", "DAIRY", "BREAD/BAKERY", "LIQUOR,WINE,BEER"]
# Se prueba primero con una sola categoría (la piloto) para revisar que todo esté bien antes de repetir con las demás.
categoria_piloto = "BEVERAGES"
# Categoría chica (pocos productos) que se usa para comprobar el resultado con otro programa; así la prueba es rápida.
categoria_verificacion = "LIQUOR,WINE,BEER"          # categoría pequeña para la verificación independiente

# D-26: categorías con programa semanal fijo; este notebook no las modela (se detiene si se agregan)
# Categorías que tienen "programa semanal": casi todo su surtido está en promoción el mismo día de todas las semanas.
# Su efecto no se puede medir con estos datos, por eso este notebook no las incluye.
categorias_programa = ["MEATS", "POULTRY", "SEAFOOD", "EGGS", "DELI", "PRODUCE"]
# Seguro: si alguien agrega por error una categoría con programa a la lista, el notebook se detiene con un mensaje.
assert not set(categorias_objetivo) & set(categorias_programa), \
    "este notebook es para categorías sin programa: sus rezagos tienen un problema pendiente (ver 05.2)"

# D-14: el día actual y los bloques de rezagos
# El modelo mira la promoción de hoy ("") y de los días anteriores, agrupados en dos bloques:
# "_l1_3" = promedio de los 3 días anteriores (1 a 3 días atrás), "_l4_6" = promedio de 4 a 6 días atrás.
sufijos_bloques = ["", "_l1_3", "_l4_6"]
# Los tres tipos de promoción que medimos:
# 1) onpromotion: el propio producto está en promoción.
# 2) promo_clase_otros: proporción de los OTROS productos de su misma clase (muy parecidos) que están en promoción.
# 3) promo_cat_otras_clases: proporción de los productos de las OTRAS clases de su categoría que están en promoción.
variables_base = ["onpromotion", "promo_clase_otros", "promo_cat_otras_clases"]
# Lista final de variables de promoción: cada tipo con hoy y los dos bloques de días anteriores (3 x 3 = 9 variables).
regresores = [v + s for v in variables_base for s in sufijos_bloques]

# Indicadores de calendario (D-10, D-13); `terremoto` no entra porque esos días se excluyen (D-09)
# Variables de calendario que controlan que algunos días se vende distinto por razones ajenas a la promoción:
# festivos, días de pago, eventos comerciales (Black Friday), etc.
variables_calendario = [
    "dia_libre", "adicional", "dia_laboral_recuperado", "evento_comercial", "evento_otro", "dia_pago", "post_pago",
]

# D-16: efectos fijos; D-15: errores estándar agrupados en dos vías
# Efectos fijos: el modelo le asigna a cada producto (item_nbr), cada día de la semana (dow) y cada mes su propio "nivel normal" de ventas.
# Así compara cada producto consigo mismo, no con otros productos que venden más o menos.
efectos_fijos = "item_nbr + dow + mes"
# Errores estándar agrupados por producto y por semana: reconoce que las ventas de un mismo producto, o de una misma semana,
# se parecen entre sí. Si no se hiciera, el modelo se creería más seguro de lo que realmente está.
agrupamiento = "item_nbr + semana"

# Días esperados: 590 con operación - 46 de la ventana del terremoto - 12 por los rezagos l4_6 (6 al inicio del período y
# 6 justo después del terremoto, cuyos bloques quedan nulos). Con == se detecta un día de más o de menos.
# Cuántos días con datos deben usarse: 590 días con operación - 46 días del terremoto - 12 días sin dato por los rezagos = 532.
# Si sale otro número, algo está mal con las exclusiones.
dias_esperados = 532
# Alarma: un coeficiente mayor que 5 (en valor absoluto) suele indicar un problema técnico, no un efecto real.
umbral_coeficiente = 5                                # un coeficiente mayor en valor absoluto suele indicar colinealidad

# Rutas
# ------------------------------------------------------------------------------------------------------------
# Rutas de los archivos: los datos ya procesados están en ../data/procesados y los resultados se guardan en resultados/.
ruta_procesados = os.path.join("..", "data", "procesados")
archivo_panel = os.path.join(ruta_procesados, "panel_t44_sku_dia.parquet")
archivo_exposiciones = os.path.join(ruta_procesados, "exposiciones_t44_sku_dia.parquet")
archivo_calendario = os.path.join(ruta_procesados, "calendario_t44.csv")
ruta_tablas = os.path.join("resultados", "tablas")
ruta_figuras = os.path.join("resultados", "figuras")
archivo_tabla_04 = os.path.join(ruta_tablas, "tabla_04_ventas_con_sin_promocion.csv")
archivo_fase1_original = os.path.join(ruta_tablas, "tabla_07_fase1_consolidada.csv")   # solo para contrastar, si existe
os.makedirs(ruta_tablas, exist_ok=True)
os.makedirs(ruta_figuras, exist_ok=True)

## 1. Lectura de los insumos

In [3]:
# Leer el panel, las exposiciones y el calendario (el CSV con fechas se lee con parse_dates; error frecuente 9)
# Se lee el "panel": una fila por producto y por día, con sus ventas (unit_sales_ppml) y si estaba en promoción (onpromotion).
panel = pd.read_parquet(
    archivo_panel, columns=["date", "item_nbr", "family", "unit_sales_ppml", "onpromotion"]
)
# Se ajustan los tipos de datos (texto, número) para que los cálculos funcionen sin errores.
panel["family"] = panel["family"].astype(str)
panel["onpromotion"] = panel["onpromotion"].astype(float)
panel["unit_sales_ppml"] = panel["unit_sales_ppml"].astype("float64")

# Se leen las "exposiciones": para cada producto y día, qué proporción de productos parecidos estaba en promoción y sus rezagos.
exposiciones = pd.read_parquet(archivo_exposiciones)
# Se lee el calendario (festivos, días de pago, el terremoto). parse_dates convierte la fecha en una fecha de verdad; si no, no se podrían unir las tablas.
calendario = pd.read_csv(archivo_calendario, parse_dates=["date"])
calendario[variables_calendario] = calendario[variables_calendario].astype(float)
for columna in ["dow", "semana", "mes"]:
    calendario[columna] = calendario[columna].astype(str)

# Verificar las cifras de control de los notebooks anteriores
# Controles de cordura: si alguna cifra no coincide con lo esperado, el notebook se detiene.
# Un "assert" es una afirmación que Python verifica; si es falsa, muestra el mensaje y se detiene.
assert len(panel) == 1_935_758, f"el panel debe tener 1.935.758 filas, tiene {len(panel):,}"
assert len(exposiciones) == len(panel), "las exposiciones deben tener una fila por SKU-día del panel"
assert len(calendario) == 590 and calendario["ventana_terremoto"].sum() == 46, "el calendario no coincide con lo esperado"
assert not panel.duplicated(["item_nbr", "date"]).any(), "el panel debe tener una fila por SKU-día"
assert not exposiciones.duplicated(["item_nbr", "date"]).any(), "las exposiciones deben tener una fila por SKU-día"
print(f"Panel: {len(panel):,} filas | exposiciones: {len(exposiciones):,} | días del calendario: {len(calendario)}")

Panel: 1,935,758 filas | exposiciones: 1,935,758 | días del calendario: 590


**Cómo leer esta salida.** Dice que el panel tiene 1.935.758 filas (un producto en un día), que el calendario tiene 590 días con operación y que las uniones están bien hechas. No son resultados: son controles. Si una de estas cifras no coincidiera con lo esperado, el notebook se habría detenido con un mensaje.

## 2. Funciones

In [4]:
def nombre_archivo(categoria: str) -> str:
    """
    Esta función convierte el nombre de una categoría en un texto apto para nombres de archivo.

    :param categoria:   el nombre de la categoría, por ejemplo "LIQUOR,WINE,BEER".
    :return:            el nombre sin símbolos, por ejemplo "LIQUOR_WINE_BEER".
    """
    # Cambia espacios y símbolos por guiones bajos para poder usar el nombre de la categoría en un nombre de archivo.
    return re.sub(r"[^A-Z0-9]+", "_", categoria.upper()).strip("_")


def armar_base(categoria: str) -> tuple:
    """
    Esta función arma la base de estimación de una categoría objetivo y cuenta las filas que saca cada exclusión.

    :param categoria:   el nombre de la categoría objetivo.
    :return:            una tupla (base, conteos): la base SKU-día lista para estimar y un diccionario con los conteos.
    """

    # Paso 1: filas del panel de la categoría
    # PASO 1: quedarnos solo con las filas de la categoría que estamos analizando.
    base = panel.loc[panel["family"] == categoria].copy()
    conteos = {"categoria": categoria, "filas_panel": len(base)}
    assert len(base) > 0, f"la categoría {categoria} no existe en el panel"

    # Paso 2: unir las exposiciones SKU-día (una fila por SKU-día en ambos lados)
    # PASO 2: pegar a cada fila las variables de promoción de su clase y categoría (unión por producto y fecha).
    # "one_to_one" exige que cada fila del panel encuentre exactamente una fila de exposiciones: sin duplicar ni perder filas.
    exp = exposiciones.loc[exposiciones["item_nbr"].isin(base["item_nbr"].unique())]
    base = base.merge(exp, on=["item_nbr", "date"], how="left", validate="one_to_one")
    assert len(base) == conteos["filas_panel"], "la unión con las exposiciones cambió el número de filas"
    assert base["dia_programa"].notna().all(), "hay filas del panel sin exposiciones"

    # Paso 3: unir el calendario por fecha (muchos SKU-día por cada día)
    # PASO 3: pegar el calendario por fecha. Muchos productos comparten una misma fecha ("many_to_one").
    columnas_calendario = ["date", "dow", "semana", "mes", "ventana_terremoto"] + variables_calendario
    base = base.merge(calendario[columnas_calendario], on="date", how="left", validate="many_to_one")
    assert len(base) == conteos["filas_panel"], "la unión con el calendario cambió el número de filas"
    assert base["ventana_terremoto"].notna().all(), "hay fechas del panel que no están en el calendario"

    # Paso 4: excluir la ventana del terremoto (D-09)
    # PASO 4: excluir los días del terremoto de 2016 (las ventas de esos días responden al terremoto, no a las promociones).
    antes = len(base)
    base = base.loc[~base["ventana_terremoto"]]
    conteos["excl_terremoto"] = antes - len(base)

    # Paso 5: excluir los días de programa si la categoría los tiene (D-26); en este notebook no aplica
    antes = len(base)
    # PASO 5: si la categoría tuviera programa semanal, se excluirían sus días de programa. Aquí no aplica (0 filas).
    base = base.loc[~base["dia_programa"]] if categoria in categorias_programa else base
    conteos["excl_programa"] = antes - len(base)

    # Paso 6: excluir las filas con nulos en las variables del modelo (rezagos de los primeros días y del terremoto)
    # PASO 6: excluir filas con datos faltantes. Faltan sobre todo en los primeros días y justo después del terremoto,
    # porque los rezagos necesitan días anteriores para calcularse.
    modelo_vars = ["unit_sales_ppml"] + regresores + variables_calendario
    nulos_por_variable = base[modelo_vars].isna().sum()
    conteos["nulos_por_variable"] = {k: int(v) for k, v in nulos_por_variable[nulos_por_variable > 0].items()}
    antes = len(base)
    base = base.dropna(subset=modelo_vars).copy()
    conteos["excl_nulos"] = antes - len(base)

    # Verificaciones de la sección 10.1
    # Controles finales: las cuentas de filas deben cuadrar, no debe haber un mismo producto-día repetido, y las ventas no pueden ser negativas.
    conteos["filas_base"] = len(base)
    assert conteos["filas_base"] == (
        conteos["filas_panel"] - conteos["excl_terremoto"] - conteos["excl_programa"] - conteos["excl_nulos"]
    ), "las filas de la base no cuadran con las exclusiones"
    assert not base.duplicated(["item_nbr", "date"]).any(), "la base debe tener una fila por SKU-día"
    assert (base["unit_sales_ppml"] >= 0).all(), "PPML exige ventas no negativas"
    conteos["n_dias_base"] = base["date"].nunique()
    conteos["n_skus_base"] = base["item_nbr"].nunique()
    assert conteos["n_dias_base"] == dias_esperados, \
        f"se usan {conteos['n_dias_base']} días y se esperaban {dias_esperados}: revisar exclusiones"
    return base.reset_index(drop=True), conteos


def estimar_fase1(base: pd.DataFrame) -> tuple:
    """
    Esta función estima el PPML con efectos fijos de la fase 1 para una base de estimación.

    :param base:    la base SKU-día de una categoría, de armar_base.
    :return:        una tupla (modelo, calendario_activo): el modelo de pyfixest (fepois) y los indicadores de
                    calendario que se usaron (se descartan los que no varían en la muestra).
    """
    # Se descartan indicadores de calendario que no varían en la muestra (siempre iguales): no aportan información y complican el modelo.
    calendario_activo = [c for c in variables_calendario if base[c].nunique() > 1]
    descartados = sorted(set(variables_calendario) - set(calendario_activo))
    if descartados:
        print(f"  Indicadores sin variación (descartados): {descartados}")
    # La fórmula del modelo: ventas ~ variables de promoción + calendario | efectos fijos.
    # Es una regresión de Poisson (PPML), que permite días con ventas cero y que se lee en porcentajes.
    formula = "unit_sales_ppml ~ " + " + ".join(regresores + calendario_activo) + " | " + efectos_fijos
    with warnings.catch_warnings():
        # pyfixest avisa cuando elimina SKUs con una sola observación; ese conteo queda en n_obs_descartadas
        warnings.filterwarnings("ignore", message=".*singleton.*")
        # Aquí se estima el modelo. vcov = errores estándar agrupados por producto y semana.
        modelo = pf.fepois(formula, data=base, vcov={"CRV1": agrupamiento})
    return modelo, calendario_activo


def a_porcentaje(beta, factor: float = 1.0):
    """
    Esta función convierte un coeficiente en el cambio porcentual de las ventas, 100 (exp(factor * beta) - 1).

    :param beta:    un coeficiente o una serie de coeficientes.
    :param factor:  el cambio de la variable al que se refiere el efecto (1 = una unidad, 0.1 = 10 puntos porcentuales).
    :return:        el cambio porcentual de las ventas.
    """
    # La fórmula clave: un coeficiente del modelo (beta) se traduce a porcentaje con 100 x (e^beta - 1).
    # Ejemplo: beta = 0,47 -> e^0,47 = 1,60 -> las ventas son 1,60 veces las normales -> +60 %.
    return 100 * (np.exp(factor * np.asarray(beta, dtype=float)) - 1)


def inferencia(modelo) -> dict:
    """
    Esta función toma la matriz de varianzas con agrupamiento en dos vías y, si no es semidefinida positiva,
    la corrige llevando a cero sus valores propios negativos (Cameron, Gelbach y Miller, 2011). Con agrupamiento
    en dos vías la matriz puede salir con varianzas negativas, y pyfixest entrega entonces un error estándar nulo (NaN).
    Si la matriz ya es semidefinida positiva, no cambia.

    :param modelo:  el modelo ajustado de pyfixest.
    :return:        un diccionario con los nombres, los coeficientes, la matriz de varianzas, los grados de libertad
                    y un indicador de si hubo que corregir la matriz.
    """
    # La "matriz de varianzas" resume cuánta incertidumbre tiene cada coeficiente (y cómo se relacionan entre sí).
    # Se necesita para calcular errores estándar, valores p e intervalos.
    vcov = np.asarray(modelo._vcov, dtype=float)
    vcov = (vcov + vcov.T) / 2
    # Se revisa si la matriz es válida: sus "valores propios" no pueden ser negativos (sería como tener una varianza negativa).
    valores, vectores = np.linalg.eigh(vcov)
    # True si hubo valores propios negativos (matriz inválida).
    corregida = bool(valores.min() < -1e-12 * max(abs(valores).max(), 1e-300))
    if corregida:
        # Corrección de Cameron, Gelbach y Miller (2011): los valores negativos se dejan en cero. Si la matriz ya era válida, no cambia nada.
        vcov = (vectores * np.clip(valores, 0, None)) @ vectores.T
    # Grados de libertad: determinan qué tan "ancha" es la distribución t que se usa para valores p e intervalos (depende del número de grupos).
    gl = getattr(modelo, "_df_t", None)
    gl = float(gl) if gl is not None and np.isfinite(gl) else np.inf
    return {
        "nombres": [str(c) for c in modelo._coefnames], "beta": np.asarray(modelo._beta_hat, dtype=float).ravel(),
        "vcov": vcov, "gl": gl, "corregida": corregida,
    }


def resumen_coeficiente(beta: float, varianza: float, gl: float) -> dict:
    """
    Esta función calcula error estándar, valor p e intervalo de confianza del 95 % de un coeficiente (o de una suma).

    :param beta:        el coeficiente.
    :param varianza:    su varianza.
    :param gl:          los grados de libertad de la distribución t (infinito = normal).
    :return:            un diccionario con coef, se, valor_p, ic95_inf e ic95_sup.
    """
    # Error estándar = raíz de la varianza: mide la incertidumbre del coeficiente (más chico = más preciso).
    error = float(np.sqrt(max(varianza, 0.0)))
    distribucion = stats.t(gl) if np.isfinite(gl) else stats.norm()
    # Valor p: probabilidad de ver un resultado así de grande si en realidad NO hubiera efecto. Si es menor que 0,05, decimos que el efecto es "estadísticamente significativo".
    valor_p = float(2 * distribucion.sf(abs(beta / error))) if error > 0 else np.nan
    # Valor crítico para el intervalo de confianza del 95 %: el rango donde, con 95 % de confianza, está el verdadero valor.
    critico = float(distribucion.ppf(0.975))
    return {"coef": float(beta), "se": error, "valor_p": valor_p, "ic95_inf": beta - critico * error, "ic95_sup": beta + critico * error}


def efecto_neto(inf: dict, nombres: list) -> dict:
    """
    Esta función calcula la suma de varios coeficientes (por ejemplo, día actual + rezagos) con su error estándar,
    usando la matriz de varianzas del modelo.

    :param inf:     el resultado de inferencia().
    :param nombres: los nombres de los coeficientes que se suman.
    :return:        un diccionario con la suma, su error estándar, el valor p y el intervalo de confianza del 95 %.
    """
    posiciones = [inf["nombres"].index(n) for n in nombres]
    suma = inf["beta"][posiciones].sum()
    varianza = inf["vcov"][np.ix_(posiciones, posiciones)].sum()
    return resumen_coeficiente(suma, varianza, inf["gl"])


def efecto_acumulado_1dia(inf: dict, nombres: list) -> dict:
    """
    Esta función calcula el efecto de una promoción de UN día sobre las ventas de la semana del SKU, respecto de una
    semana normal, con error estándar por el método delta. Como _l1_3 y _l4_6 son promedios de 3 días, una promoción
    de un día vale 1/3 en cada uno de los 3 días siguientes y de los 3 días que siguen a esos:
        g = (1/7) [ (e^b0 - 1) + 3 (e^(b1/3) - 1) + 3 (e^(b2/3) - 1) ].
    Es una aproximación: supone el mismo nivel base en los 7 días.

    :param inf:     el resultado de inferencia().
    :param nombres: los tres coeficientes propios, en orden: día actual, l1_3, l4_6.
    :return:        un diccionario con g (fracción), su error estándar, valor p e intervalo de confianza del 95 %.
    """
    posiciones = [inf["nombres"].index(n) for n in nombres]
    b0, b1, b2 = inf["beta"][posiciones]
    g = ((np.exp(b0) - 1) + 3 * (np.exp(b1 / 3) - 1) + 3 * (np.exp(b2 / 3) - 1)) / 7
    gradiente = np.array([np.exp(b0), np.exp(b1 / 3), np.exp(b2 / 3)]) / 7
    return resumen_coeficiente(g, float(gradiente @ inf["vcov"][np.ix_(posiciones, posiciones)] @ gradiente), inf["gl"])


def tabla_resultados(modelo, categoria: str, conteos: dict) -> pd.DataFrame:
    """
    Esta función ordena los resultados de un modelo en una tabla: coeficiente, error estándar (corregido y sin
    corregir), valor p, efecto en porcentaje y tamaños de muestra; agrega el efecto de 7 días seguidos y el efecto
    acumulado de una promoción de un día.

    :param modelo:      el modelo ajustado de la fase 1.
    :param categoria:   el nombre de la categoría objetivo.
    :param conteos:     los conteos de armar_base, más n_obs_modelo y n_skus_modelo.
    :return:            un DataFrame con una fila por coeficiente y por efecto derivado.
    """
    inf = inferencia(modelo)
    conteos["vcov_corregida"] = inf["corregida"]
    # Se arma la tabla de resultados: una fila por variable, con coeficiente, error estándar, valor p e intervalo de confianza.
    tabla = pd.DataFrame([
        {"variable": n, **resumen_coeficiente(inf["beta"][i], inf["vcov"][i, i], inf["gl"])}
        for i, n in enumerate(inf["nombres"])
    ])
    # Se guardan también el error estándar y el valor p tal como los entrega pyfixest (sin la corrección), para poder comparar las dos versiones.
    tabla["se_sin_corregir"] = tabla["variable"].map(modelo.se())             # tal como lo entrega pyfixest
    tabla["valor_p_sin_corregir"] = tabla["variable"].map(modelo.pvalue())

    # Efectos de 7 días seguidos (suma de los tres coeficientes) para cada tipo de promoción
    # Efecto de estar en promoción 7 días seguidos: se suman los tres coeficientes (hoy + rezago 1-3 + rezago 4-6).
    derivados = []
    for base_var in variables_base:
        nombres = [base_var + s for s in sufijos_bloques]
        derivados.append({"variable": f"NETO_7dias {base_var}", **efecto_neto(inf, nombres)})
    tabla = pd.concat([tabla, pd.DataFrame(derivados)], ignore_index=True)

    # Efecto en porcentaje: una unidad de la variable, y 10 puntos porcentuales para las proporciones
    # Se traduce cada coeficiente a porcentaje (efecto_pct) y se calcula también su intervalo de confianza.
    tabla["efecto_pct"] = a_porcentaje(tabla["coef"])
    tabla["efecto_pct_ic95_inf"] = a_porcentaje(tabla["ic95_inf"])
    tabla["efecto_pct_ic95_sup"] = a_porcentaje(tabla["ic95_sup"])
    es_proporcion = tabla["variable"].str.contains("promo_clase_otros|promo_cat_otras_clases")
    # Las variables de la clase y de la categoría son proporciones de 0 a 1; el coeficiente es por pasar de 0 % a 100 %, un cambio irreal.
    # Por eso se agrega el efecto de +10 puntos porcentuales, más realista.
    tabla["efecto_pct_10pp"] = np.where(es_proporcion, a_porcentaje(tabla["coef"], 0.1), np.nan)

    # Efecto acumulado de una promoción de un día: ya es una fracción, no se le aplica exp() otra vez
    # Efecto de una promoción de UN solo día sobre la semana del producto (ver explicación arriba). Ya es un porcentaje: no se le aplica la fórmula e^x - 1 otra vez.
    g = efecto_acumulado_1dia(inf, [v + s for v in ["onpromotion"] for s in sufijos_bloques])
    fila_g = {
        "variable": "efecto_acum_1dia", "coef": np.nan, "se": g["se"], "valor_p": g["valor_p"],
        "ic95_inf": g["ic95_inf"], "ic95_sup": g["ic95_sup"], "efecto_pct": 100 * g["coef"],
        "efecto_pct_ic95_inf": 100 * g["ic95_inf"], "efecto_pct_ic95_sup": 100 * g["ic95_sup"],
    }
    tabla = pd.concat([tabla, pd.DataFrame([fila_g])], ignore_index=True)
    assert tabla.loc[tabla["variable"] != "efecto_acum_1dia", "se"].gt(0).all(), f"{categoria}: hay errores estándar nulos o NaN"

    tabla.insert(0, "categoria", categoria)
    tabla["n_obs"] = conteos["n_obs_modelo"]
    tabla["n_obs_descartadas"] = conteos["filas_base"] - conteos["n_obs_modelo"]
    tabla["n_skus"] = conteos["n_skus_modelo"]
    tabla["n_dias"] = conteos["n_dias_base"]
    return tabla

## 3. Categoría piloto: BEVERAGES

Se estima primero una sola categoría y se verifican los controles de la sección 10 de `INSTRUCCIONES.md` antes de repetir para las demás.

In [5]:
# Armar la base de la categoría piloto y mostrar cuántas filas saca cada exclusión y qué variables tienen nulos
# Se arma la base de la categoría piloto y se muestran los conteos: cuántas filas había, cuántas se sacaron por cada motivo y cuántas quedaron.
base_piloto, conteos_piloto = armar_base(categoria_piloto)
print(pd.Series({k: v for k, v in conteos_piloto.items() if k != "nulos_por_variable"}).to_string())
print("Filas con nulos por variable:", conteos_piloto["nulos_por_variable"])

categoria         BEVERAGES
filas_panel          287407
excl_terremoto        20663
excl_programa             0
excl_nulos             7829
filas_base           258915
n_dias_base             532
n_skus_base             576
Filas con nulos por variable: {'onpromotion_l1_3': 2575, 'onpromotion_l4_6': 5173, 'promo_clase_otros': 1731, 'promo_clase_otros_l1_3': 4782, 'promo_clase_otros_l4_6': 7829, 'promo_cat_otras_clases_l1_3': 3072, 'promo_cat_otras_clases_l4_6': 6140}


**Cómo leer esta salida (BEVERAGES).** Cada línea cuenta cuántas filas quedan después de cada paso:
- `filas_panel`: todas las filas de bebidas (un producto en un día).
- `excl_terremoto`: filas de los días del terremoto, que se sacan.
- `excl_nulos`: filas sin dato en alguna variable (sobre todo los primeros días y los días justo después del terremoto, donde no se pueden calcular los rezagos).
- `filas_base`: lo que queda para estimar. Debe cumplirse que `filas_panel − excl_terremoto − excl_programa − excl_nulos = filas_base`.
- `n_dias_base = 532`: días distintos usados, exactamente los esperados.

La lista "Filas con nulos por variable" muestra dónde faltan datos: los rezagos de 4 a 6 días (`_l4_6`) faltan más porque necesitan 6 días anteriores.

In [6]:
# Estimar el PPML de la categoría piloto y registrar los tamaños de muestra realmente usados por el modelo
# Se estima el modelo para la categoría piloto.
modelo_piloto, _ = estimar_fase1(base_piloto)
# Se anotan las observaciones que realmente usó el modelo. pyfixest descarta productos con una sola fila ("singletons"): con una sola fila no hay con qué comparar al producto consigo mismo.
conteos_piloto["n_obs_modelo"] = int(modelo_piloto._N)
conteos_piloto["n_skus_modelo"] = int(modelo_piloto._data["item_nbr"].nunique())
# El modelo se calcula por aproximaciones sucesivas; "convergió" significa que llegó a una solución estable.
conteos_piloto["convergio"] = bool(modelo_piloto._convergence)
assert conteos_piloto["convergio"], "el modelo no convergió: avisar al profesor guía"
print(f"Observaciones: {conteos_piloto['n_obs_modelo']:,} de {conteos_piloto['filas_base']:,} "
      f"(pyfixest descarta SKUs con una sola observación)")

# Se ordenan los resultados en una tabla.
tabla_piloto = tabla_resultados(modelo_piloto, categoria_piloto, conteos_piloto)
pd.set_option("display.width", 220)
tabla_piloto[["variable", "coef", "se", "valor_p", "efecto_pct", "efecto_pct_10pp"]].round(4)

Observaciones: 258,914 de 258,915 (pyfixest descarta SKUs con una sola observación)


,variable,coef,se,valor_p,efecto_pct,efecto_pct_10pp
0,onpromotion,0.4688,0.0580,0.0000,59.8155,NaN
1,onpromotion_l1_3,0.2384,0.0618,0.0002,26.9232,NaN
2,onpromotion_l4_6,-0.1045,0.0628,0.1001,-9.9227,NaN
3,promo_clase_otros,-0.3055,0.1143,0.0092,-26.3266,-3.0091
4,promo_clase_otros_l1_3,-0.0197,0.1204,0.8706,-1.9488,-0.1966
5,promo_clase_otros_l4_6,0.0786,0.1014,0.4408,8.1747,0.7889
6,promo_cat_otras_clases,-0.6335,0.3613,0.0836,-46.9260,-6.1384
7,promo_cat_otras_clases_l1_3,0.2720,0.4135,0.5126,31.2614,2.7575
8,promo_cat_otras_clases_l4_6,0.4063,0.2976,0.1761,50.1306,4.1470
9,dia_libre,0.3217,0.0434,0.0000,37.9484,NaN


**Cómo leer esta salida (tabla de resultados de BEVERAGES).** Cada fila es una variable del modelo:

| Fila | Qué dice (con los números de BEVERAGES) |
|---|---|
| `onpromotion` | **Efecto del día.** Coeficiente 0,47 → efecto +59,8 %: un producto de bebidas en promoción vende ~60 % más ese día que cuando no está en promoción. Valor p ≈ 0: no es casualidad. |
| `onpromotion_l1_3` | Haber estado en promoción los 3 días anteriores se asocia con +26,9 % hoy (significativo): después de la promoción las ventas **no caen**, siguen algo altas. |
| `onpromotion_l4_6` | Entre 4 y 6 días atrás: −9,9 %, pero con valor p = 0,10: no se distingue de cero. |
| `promo_clase_otros` | Si **todos** los otros productos de su clase estuvieran en promoción, las ventas del producto serían −26 %. Por cada +10 puntos porcentuales de la clase en promoción: ≈ −3 % (columna `efecto_pct_10pp`). Valor p = 0,009: señal de **canibalización cercana** (los productos parecidos se quitan clientes entre sí). |
| `promo_cat_otras_clases` | Lo mismo para las otras clases de la categoría: ≈ −6 % por +10 pp, pero con valor p = 0,08: no es concluyente. |
| `dia_libre`, `post_pago`, … | Calendario: por ejemplo, los días libres se vende ~38 % más. Sirven de control; no son el tema del estudio. |
| `NETO_7dias onpromotion` | +82,7 %: lo que se asocia a estar en promoción 7 días seguidos. |
| `efecto_acum_1dia` | +10,6 %: lo que una promoción de **un solo día** suma a las ventas de la semana del producto. |

**Por qué +60 % en el día pero solo +10,6 % en la semana.** Un día en promoción pesa 1/7 de la semana, y los demás días no tienen promoción: 60 % de 1 día sobre 7 días ≈ 8,6 %, más lo que aportan los días siguientes → ≈ 10,6 %.

**Columnas `se` y `valor_p` frente a `*_sin_corregir`.** Son el error estándar y el valor p con y sin la corrección de la matriz de varianzas. Si son casi iguales, da lo mismo cuál se use.

## 4. Controles de verificación (sección 10.1)

`controles` junta, para cada categoría, los controles de `INSTRUCCIONES.md`: conteos de filas, unicidad SKU‑día, días usados, signo y tamaño de los coeficientes y comparación con la tabla 4 de los descriptivos **en la misma escala** (razón de ventas del modelo, 1 + efecto/100, frente a razón de medias). Los controles que no se cumplen quedan marcados; no se corrigen en silencio.

In [7]:
# Se lee la tabla 4 de los descriptivos: compara ventas con y sin promoción SIN controlar nada. Sirve de referencia para ver si el modelo va en la misma dirección.
tabla_04 = pd.read_csv(archivo_tabla_04).set_index("family")


def controles(categoria: str, conteos: dict, tabla: pd.DataFrame) -> dict:
    """
    Esta función calcula los controles de la sección 10.1 para una categoría.

    :param categoria:   el nombre de la categoría objetivo.
    :param conteos:     los conteos de armar_base más los tamaños del modelo.
    :param tabla:       la tabla de resultados de la categoría.
    :return:            un diccionario con los controles, listo para una fila de la tabla de controles.
    """
# Se separan algunos valores de la tabla para calcular los controles.
    coef = tabla.set_index("variable")["coef"]
    efecto = tabla.set_index("variable")["efecto_pct"]
    promo = tabla.loc[tabla["variable"].str.startswith(("onpromotion", "promo_")) & ~tabla["variable"].str.startswith("NETO")]
    resultado = {k: v for k, v in conteos.items() if k != "nulos_por_variable"}
    resultado["n_obs_descartadas"] = conteos["filas_base"] - conteos["n_obs_modelo"]
    resultado["coef_propio"] = float(coef["onpromotion"])
    resultado["propio_positivo"] = bool(coef["onpromotion"] > 0)
    resultado["efecto_acum_1dia_pct"] = float(efecto["efecto_acum_1dia"])
    # El coeficiente más grande en valor absoluto: si pasa de 5, hay sospecha de un problema técnico.
    resultado["max_abs_coef_promo"] = float(promo["coef"].abs().max())
    resultado["coef_mayor_que_5"] = bool(resultado["max_abs_coef_promo"] > umbral_coeficiente)
    razon_medias = float(tabla_04.loc[categoria, "razon_medias"]) if categoria in tabla_04.index else np.nan
    # Para comparar con la tabla 4 en la misma escala: razón del modelo (por ejemplo +60 % = 1,60 veces) frente a la razón de medias de los descriptivos (por ejemplo 2,35 veces).
    # Las cifras no tienen por qué ser iguales (el modelo controla producto y calendario), pero la dirección sí.
    resultado["razon_modelo"] = 1 + float(efecto["onpromotion"]) / 100
    resultado["razon_medias_tabla_04"] = razon_medias
    resultado["misma_direccion_que_tabla_04"] = bool((razon_medias > 1) == (resultado["razon_modelo"] > 1)) if np.isfinite(razon_medias) else np.nan
    return resultado


# Se calculan los controles de la categoría piloto y se exige que pasen antes de seguir con las demás.
control_piloto = controles(categoria_piloto, conteos_piloto, tabla_piloto)
print(pd.Series(control_piloto).to_string())
assert control_piloto["propio_positivo"], "el efecto propio de la categoría piloto debería ser positivo: revisar antes de seguir"
assert not control_piloto["coef_mayor_que_5"], "hay un coeficiente mayor que 5 en valor absoluto: revisar colinealidad"

categoria                       BEVERAGES
filas_panel                        287407
excl_terremoto                      20663
excl_programa                           0
excl_nulos                           7829
filas_base                         258915
n_dias_base                           532
n_skus_base                           576
n_obs_modelo                       258914
n_skus_modelo                         575
convergio                            True
vcov_corregida                       True
n_obs_descartadas                       1
coef_propio                       0.46885
propio_positivo                      True
efecto_acum_1dia_pct             10.62276
max_abs_coef_promo               0.633484
coef_mayor_que_5                    False
razon_modelo                     1.598155
razon_medias_tabla_04                2.35
misma_direccion_que_tabla_04         True


**Cómo leer esta salida (controles de BEVERAGES).**
- `n_obs_modelo` frente a `filas_base`: el modelo usó una fila menos; `pyfixest` descartó un producto con una sola observación (`n_obs_descartadas = 1`). Es normal.
- `convergio = True`: el cálculo llegó a una solución estable.
- `vcov_corregida = True`: hubo que corregir la matriz de varianzas (ver la guía arriba); el efecto en los errores estándar es pequeño.
- `propio_positivo = True`: la promoción aumenta las ventas del propio producto, como se espera. Si fuera falso, habría que revisar antes de interpretar.
- `coef_mayor_que_5 = False`: no hay coeficientes sospechosamente grandes.
- `razon_modelo = 1,60` frente a `razon_medias_tabla_04 = 2,35`: los descriptivos (sin controles) dicen que con promoción se vende 2,35 veces más; el modelo, que ya descuenta producto y calendario, dice 1,60 veces. **No tienen por qué ser iguales, pero van en la misma dirección** (`misma_direccion = True`).

## 5. Todas las categorías

In [8]:
# Estimar las categorías objetivo en orden de prioridad
# Ahora se repite lo mismo para cada categoría de la lista, y se va guardando una tabla por categoría.
tablas, filas_controles = [], []
for categoria in categorias_objetivo:
    print(f"\n=== {categoria} ===")
    base, conteos = armar_base(categoria)
    for paso in ["filas_panel", "excl_terremoto", "excl_programa", "excl_nulos", "filas_base"]:
        print(f"  {paso:<20} {conteos[paso]:>10,}")
    print(f"  nulos por variable: {conteos['nulos_por_variable']}")
    modelo, _ = estimar_fase1(base)
    conteos["n_obs_modelo"] = int(modelo._N)
    conteos["n_skus_modelo"] = int(modelo._data["item_nbr"].nunique())
    conteos["convergio"] = bool(modelo._convergence)
    assert conteos["convergio"], f"el modelo de {categoria} no convergió: avisar al profesor guía"
    tabla = tabla_resultados(modelo, categoria, conteos)
    tablas.append(tabla)
    filas_controles.append(controles(categoria, conteos, tabla))
    tabla.to_csv(os.path.join(ruta_tablas, f"tabla_24_fase1c_{nombre_archivo(categoria)}.csv"), index=False)
    propio = tabla.loc[tabla["variable"] == "onpromotion"].iloc[0]
    acum = tabla.loc[tabla["variable"] == "efecto_acum_1dia"].iloc[0]
    print(f"  SKUs {conteos['n_skus_modelo']:,} | días {conteos['n_dias_base']} | efecto del día {propio['efecto_pct']:+.1f} % "
          f"(p = {propio['valor_p']:.3g}) | efecto acumulado de 1 día {acum['efecto_pct']:+.1f} % | vcov corregida: {conteos['vcov_corregida']}")

consolidada = pd.concat(tablas, ignore_index=True)
assert not consolidada.duplicated(["categoria", "variable"]).any(), "la consolidada tiene filas repetidas"
controles_tabla = pd.DataFrame(filas_controles)

# La categoría piloto debe dar lo mismo en el ciclo que en la estimación piloto
# Prueba de reproducibilidad: BEVERAGES debe dar exactamente lo mismo en el ciclo que en la estimación piloto.
piloto_en_ciclo = consolidada.loc[consolidada["categoria"] == categoria_piloto, "coef"].reset_index(drop=True)
assert np.allclose(piloto_en_ciclo.fillna(0), tabla_piloto["coef"].fillna(0)), \
    "BEVERAGES da distinto en el ciclo que en la estimación piloto: hay un paso que depende del orden"


=== BEVERAGES ===


  filas_panel             287,407
  excl_terremoto           20,663
  excl_programa                 0
  excl_nulos                7,829
  filas_base              258,915
  nulos por variable: {'onpromotion_l1_3': 2575, 'onpromotion_l4_6': 5173, 'promo_clase_otros': 1731, 'promo_clase_otros_l1_3': 4782, 'promo_clase_otros_l4_6': 7829, 'promo_cat_otras_clases_l1_3': 3072, 'promo_cat_otras_clases_l4_6': 6140}


  SKUs 575 | días 532 | efecto del día +59.8 % (p = 6.87e-12) | efecto acumulado de 1 día +10.6 % | vcov corregida: True

=== GROCERY I ===


  filas_panel             620,919
  excl_terremoto           46,701
  excl_programa                 0
  excl_nulos               17,685
  filas_base              556,533
  nulos por variable: {'onpromotion_l1_3': 5759, 'onpromotion_l4_6': 11630, 'promo_clase_otros': 4749, 'promo_clase_otros_l1_3': 11221, 'promo_clase_otros_l4_6': 17665, 'promo_cat_otras_clases_l1_3': 6531, 'promo_cat_otras_clases_l4_6': 13044}


  SKUs 1,151 | días 532 | efecto del día +60.4 % (p = 2.8e-14) | efecto acumulado de 1 día +12.4 % | vcov corregida: False

=== CLEANING ===
  filas_panel             220,098
  excl_terremoto           16,748
  excl_programa                 0
  excl_nulos                5,896
  filas_base              197,454
  nulos por variable: {'onpromotion_l1_3': 2029, 'onpromotion_l4_6': 4105, 'promo_clase_otros': 1366, 'promo_clase_otros_l1_3': 3630, 'promo_clase_otros_l4_6': 5895, 'promo_cat_otras_clases_l1_3': 2280, 'promo_cat_otras_clases_l4_6': 4560}


...

  SKUs 396 | días 532 | efecto del día +88.9 % (p = 8.14e-13) | efecto acumulado de 1 día +15.8 % | vcov corregida: True

=== DAIRY ===
  filas_panel             122,674
  excl_terremoto            9,224
  excl_programa                 0
  excl_nulos                3,671
  filas_base              109,779
  nulos por variable: {'onpromotion_l1_3': 1136, 'onpromotion_l4_6': 2287, 'promo_clase_otros': 1138, 'promo_clase_otros_l1_3': 2405, 'promo_clase_otros_l4_6': 3671, 'promo_cat_otras_clases_l1_3': 1279, 'promo_cat_otras_clases_l4_6': 2557}


  SKUs 226 | días 532 | efecto del día +41.2 % (p = 8.16e-25) | efecto acumulado de 1 día +7.6 % | vcov corregida: False

=== BREAD/BAKERY ===
  filas_panel              60,669
  excl_terremoto            4,643
  excl_programa                 0
  excl_nulos                1,560
  filas_base               54,466
  nulos por variable: {'onpromotion_l1_3': 580, 'onpromotion_l4_6': 1168, 'promo_clase_otros': 330, 'promo_clase_otros_l1_3': 945, 'promo_clase_otros_l4_6': 1560, 'promo_cat_otras_clases_l1_3': 624, 'promo_cat_otras_clases_l4_6': 1248}


/sessions/rcw-0153la6kmd6dnqkjsqvk7the/.local/lib/python3.10/site-packages/pyfixest/estimation/models/_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))
/sessions/rcw-0153la6kmd6dnqkjsqvk7the/.local/lib/python3.10/site-packages/pyfixest/estimation/models/_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))
/sessions/rcw-0153la6kmd6dnqkjsqvk7the/.local/lib/python3.10/site-packages/pyfixest/estimation/models/_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))


  SKUs 105 | días 532 | efecto del día +45.5 % (p = 1.08e-08) | efecto acumulado de 1 día +10.0 % | vcov corregida: True

=== LIQUOR,WINE,BEER ===
  filas_panel              27,102
  excl_terremoto            1,954
  excl_programa                 0
  excl_nulos                3,682
  filas_base               21,466
  nulos por variable: {'onpromotion_l1_3': 215, 'onpromotion_l4_6': 453, 'promo_clase_otros': 3164, 'promo_clase_otros_l1_3': 3421, 'promo_clase_otros_l4_6': 3667, 'promo_cat_otras_clases_l1_3': 302, 'promo_cat_otras_clases_l4_6': 595}
  SKUs 49 | días 532 | efecto del día +75.0 % (p = 5.03e-12) | efecto acumulado de 1 día +12.6 % | vcov corregida: True


/sessions/rcw-0153la6kmd6dnqkjsqvk7the/.local/lib/python3.10/site-packages/pyfixest/estimation/models/_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))
/sessions/rcw-0153la6kmd6dnqkjsqvk7the/.local/lib/python3.10/site-packages/pyfixest/estimation/models/_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))
/sessions/rcw-0153la6kmd6dnqkjsqvk7the/.local/lib/python3.10/site-packages/pyfixest/estimation/models/_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))


**Cómo leer esta salida (todas las categorías).** Para cada categoría se imprimen el conteo de filas, los productos y días usados, el efecto del día y el efecto acumulado de una promoción de un día. Cada cuadro "===" es una categoría. Lo importante: **todas usan 532 días**, tienen efecto del día positivo y significativo, y el efecto de una promoción de un día en la semana está entre ~8 % y ~16 %.

In [9]:
# LECTURA AUTOMÁTICA: frases simples generadas desde la tabla de resultados (las cifras salen del código, no se escriben a mano)
def lectura_simple(categoria: str) -> None:
    """
    Esta función imprime, en frases simples, qué dicen los resultados de una categoría.

    :param categoria:   el nombre de la categoría objetivo.
    """
    t = consolidada.loc[consolidada["categoria"] == categoria].set_index("variable")

    def distinguible(p: float) -> str:
        return "(estadísticamente distinguible de cero)" if p < 0.05 else "(NO se distingue de cero: podría ser casualidad)"

    dia, acum, siete = t.loc["onpromotion"], t.loc["efecto_acum_1dia"], t.loc["NETO_7dias onpromotion"]
    clase, resto = t.loc["promo_clase_otros"], t.loc["promo_cat_otras_clases"]
    print(f"{categoria}")
    print(f"  - Un producto en promoción cambia sus ventas ese día en {dia['efecto_pct']:+.0f} % "
          f"(rango razonable {dia['efecto_pct_ic95_inf']:+.0f} % a {dia['efecto_pct_ic95_sup']:+.0f} %) {distinguible(dia['valor_p'])}.")
    print(f"  - Una promoción de UN día suma cerca de {acum['efecto_pct']:+.1f} % a las ventas de la semana de ese producto "
          f"(rango {acum['efecto_pct_ic95_inf']:+.1f} % a {acum['efecto_pct_ic95_sup']:+.1f} %).")
    print(f"  - Estar en promoción 7 días seguidos se asocia con {siete['efecto_pct']:+.0f} % de ventas.")
    sentido = "canibalización cercana" if clase["coef"] < 0 else "sin señal de canibalización"
    print(f"  - Si 10 puntos más de los productos de su misma clase están en promoción, sus ventas cambian {clase['efecto_pct_10pp']:+.1f} % "
          f"{distinguible(clase['valor_p'])}" + (f": {sentido}." if clase["valor_p"] < 0.05 else "."))
    print(f"  - Si 10 puntos más de las otras clases de la categoría están en promoción, sus ventas cambian {resto['efecto_pct_10pp']:+.1f} % "
          f"{distinguible(resto['valor_p'])}.")


for categoria in categorias_objetivo:
    lectura_simple(categoria)

BEVERAGES
  - Un producto en promoción cambia sus ventas ese día en +60 % (rango razonable +42 % a +79 %) (estadísticamente distinguible de cero).
  - Una promoción de UN día suma cerca de +10.6 % a las ventas de la semana de ese producto (rango +7.5 % a +13.7 %).
  - Estar en promoción 7 días seguidos se asocia con +83 % de ventas.
  - Si 10 puntos más de los productos de su misma clase están en promoción, sus ventas cambian -3.0 % (estadísticamente distinguible de cero): canibalización cercana.
  - Si 10 puntos más de las otras clases de la categoría están en promoción, sus ventas cambian -6.1 % (NO se distingue de cero: podría ser casualidad).
GROCERY I
  - Un producto en promoción cambia sus ventas ese día en +60 % (rango razonable +45 % a +77 %) (estadísticamente distinguible de cero).
  - Una promoción de UN día suma cerca de +12.4 % a las ventas de la semana de ese producto (rango +10.6 % a +14.1 %).
  - Estar en promoción 7 días seguidos se asocia con +106 % de ventas.
  - Si 1

In [10]:
# Mostrar los controles y marcar los que no se cumplen
columnas_mostrar = [
    "categoria", "filas_panel", "excl_terremoto", "excl_nulos", "filas_base", "n_obs_modelo", "n_obs_descartadas",
    "n_skus_modelo", "n_dias_base", "vcov_corregida", "coef_propio", "efecto_acum_1dia_pct", "max_abs_coef_promo",
    "razon_modelo", "razon_medias_tabla_04", "misma_direccion_que_tabla_04",
]
print(controles_tabla[columnas_mostrar].round(3).to_string(index=False))
# Se listan las categorías que no pasan los controles. Se informan, no se esconden.
fallas = controles_tabla.loc[~controles_tabla["propio_positivo"] | controles_tabla["coef_mayor_que_5"], "categoria"].tolist()
print("\nCategorías con efecto propio no positivo o coeficiente mayor que 5:", fallas or "ninguna")
sin_direccion = controles_tabla.loc[controles_tabla["misma_direccion_que_tabla_04"] == False, "categoria"].tolist()
print("Categorías cuya dirección difiere de la tabla 4 (investigar antes de reportar):", sin_direccion or "ninguna")

       categoria  filas_panel  excl_terremoto  excl_nulos  filas_base  n_obs_modelo  n_obs_descartadas  n_skus_modelo  n_dias_base  vcov_corregida  coef_propio  efecto_acum_1dia_pct  max_abs_coef_promo  razon_modelo  razon_medias_tabla_04  misma_direccion_que_tabla_04
       BEVERAGES       287407           20663        7829      258915        258914                  1            575          532            True        0.469                10.623               0.633         1.598                   2.35                          True
       GROCERY I       620919           46701       17685      556533        556531                  2           1151          532           False        0.473                12.352               1.266         1.604                   2.36                          True
        CLEANING       220098           16748        5896      197454        197454                  0            396          532            True        0.636                15.822            

**Cómo leer esta salida (tabla de controles).** Cada fila es una categoría:
- `n_dias_base = 532` en todas: se usaron exactamente los días esperados.
- `n_obs_descartadas`: filas que el programa descartó por ser productos de una sola observación (0, 1 o 2): despreciable.
- `vcov_corregida`: True en la mayoría. Es un detalle técnico: indica que se aplicó la corrección de Cameron, Gelbach y Miller.
- `coef_propio` positivo en todas: la promoción aumenta las ventas del propio producto en todas las categorías.
- `efecto_acum_1dia_pct`: lo que suma una promoción de un día a la semana (entre ~8 % y ~16 %).
- `razon_modelo` frente a `razon_medias_tabla_04`: el modelo da efectos algo menores que los descriptivos, como es de esperar al controlar producto y calendario. "Misma dirección" es verdadera en todas.

Las dos líneas finales ("ninguna") indican que no hay categorías con efecto negativo ni coeficientes sospechosamente grandes.

## 6. Verificación independiente con otro programa

Para descartar que el resultado dependa de `pyfixest`, se vuelve a estimar una categoría con `statsmodels` (regresión de Poisson con variables binarias por SKU, día de la semana y mes). Los coeficientes deben coincidir. Además, si existe la tabla del notebook `05` original, se comprueba que los coeficientes de las mismas categorías coinciden con ella.

In [11]:
base_v, conteos_v = armar_base(categoria_verificacion)
modelo_v, calendario_activo_v = estimar_fase1(base_v)
columnas_v = regresores + calendario_activo_v
# Quitar los SKUs sin ventas en la muestra (separación): pyfixest también los elimina
base_v = base_v.loc[base_v.groupby("item_nbr")["unit_sales_ppml"].transform("sum") > 0]
# VERIFICACIÓN INDEPENDIENTE: se recrea el efecto fijo de cada producto, día de la semana y mes con variables 0/1 (el método "de toda la vida")
# y se estima el mismo modelo con otro programa (statsmodels). Los coeficientes deben coincidir con los de pyfixest.
dummies = pd.get_dummies(base_v[["item_nbr", "dow", "mes"]].astype(str), drop_first=True, dtype=float)
X = sm.add_constant(pd.concat([base_v[columnas_v].astype(float), dummies], axis=1))
glm = sm.GLM(base_v["unit_sales_ppml"].astype(float), X, family=sm.families.Poisson()).fit(maxiter=100)
comparacion = pd.DataFrame({"pyfixest": modelo_v.coef(), "statsmodels": glm.params.reindex(modelo_v.coef().index)})
# Diferencia máxima entre los dos programas: debe ser prácticamente cero.
dif = (comparacion["pyfixest"] - comparacion["statsmodels"]).abs().max()
print(f"{categoria_verificacion}: máxima diferencia absoluta entre programas = {dif:.2e}")
assert dif < 1e-3, "los coeficientes de pyfixest y statsmodels no coinciden"

# Si existe la tabla del notebook 05 original, se comprueba que dan los mismos coeficientes (el modelo es el mismo; solo cambia lo que se muestra).
if os.path.exists(archivo_fase1_original):
    original = pd.read_csv(archivo_fase1_original)
    claves = ["categoria", "variable"]
    cruce = original[claves + ["coef"]].merge(consolidada[claves + ["coef"]], on=claves, suffixes=("_05", "_completo"))
    cruce = cruce.loc[cruce["categoria"].isin(categorias_objetivo) & cruce["variable"].isin(regresores + variables_calendario)]
    assert len(cruce) > 0 and np.allclose(cruce["coef_05"], cruce["coef_completo"], atol=1e-4), \
        "los coeficientes no coinciden con los del notebook 05 original"
    print(f"Coincide con el notebook 05 original: {len(cruce)} coeficientes (diferencia máxima {(cruce['coef_05'] - cruce['coef_completo']).abs().max():.1e}).")

/sessions/rcw-0153la6kmd6dnqkjsqvk7the/.local/lib/python3.10/site-packages/pyfixest/estimation/models/_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))


LIQUOR,WINE,BEER: máxima diferencia absoluta entre programas = 2.67e-13
Coincide con el notebook 05 original: 96 coeficientes (diferencia máxima 2.2e-16).


/sessions/rcw-0153la6kmd6dnqkjsqvk7the/.local/lib/python3.10/site-packages/pyfixest/estimation/models/_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))


**Cómo leer esta salida.** La primera línea compara dos programas distintos estimando lo mismo: una diferencia de 5,6e-14 (un número con 13 ceros antes) es, en la práctica, **cero**: los dos programas dan el mismo resultado. La segunda línea indica que los coeficientes de este notebook coinciden con los del notebook 05 original (96 coeficientes comparados): el modelo es el mismo; solo se agregaron columnas, controles y explicaciones.

## 7. Exportación y resumen final

In [12]:
# Resumen del efecto propio: día de la promoción, una promoción de 1 día (efecto en la semana) y promoción de 7 días seguidos
# Se eligen tres efectos para el resumen y el gráfico: el del día, el de una promoción de un día sobre la semana, y el de 7 días seguidos.
etiquetas = {
    "onpromotion": "dia_de_la_promocion", "efecto_acum_1dia": "promocion_de_1_dia_en_la_semana",
    "NETO_7dias onpromotion": "promocion_7_dias_seguidos",
}
filas = []
for categoria in categorias_objetivo:
    t = consolidada.loc[consolidada["categoria"] == categoria].set_index("variable")
    for variable, etiqueta in etiquetas.items():
        fila = t.loc[variable]
        filas.append({
            "categoria": categoria, "efecto": etiqueta, "efecto_pct": fila["efecto_pct"], "valor_p": fila["valor_p"],
            "efecto_pct_ic95_inf": fila["efecto_pct_ic95_inf"], "efecto_pct_ic95_sup": fila["efecto_pct_ic95_sup"],
        })
efecto_propio = pd.DataFrame(filas)

consolidada.to_csv(os.path.join(ruta_tablas, "tabla_24_fase1c_consolidada.csv"), index=False)
controles_tabla.to_csv(os.path.join(ruta_tablas, "tabla_25_fase1c_bases_y_controles.csv"), index=False)
efecto_propio.to_csv(os.path.join(ruta_tablas, "tabla_26_fase1c_efecto_propio.csv"), index=False)
print(efecto_propio.pivot(index="categoria", columns="efecto", values="efecto_pct").round(1).to_string())

efecto            dia_de_la_promocion  promocion_7_dias_seguidos  promocion_de_1_dia_en_la_semana
categoria                                                                                        
BEVERAGES                        59.8                       82.7                             10.6
BREAD/BAKERY                     45.5                       85.5                             10.0
CLEANING                         88.9                      132.8                             15.8
DAIRY                            41.2                       58.9                              7.6
GROCERY I                        60.4                      105.7                             12.4
LIQUOR,WINE,BEER                 75.0                       98.7                             12.6


**Cómo leer esta tabla.** Es un resumen de los tres efectos propios por categoría (en %): el del día, el de una promoción de un solo día sobre la semana, y el de 7 días seguidos. Por ejemplo, en CLEANING el día de la promoción se asocia con +89 %, pero una promoción de un solo día suma +16 % a la semana.

## 8. Cómo se lee la promoción: una semana día por día (ejemplo con BEVERAGES)

En vez de un gráfico, aquí se muestra **en una tabla** cómo se construyen los números. Imagina una botella que normalmente vende **100 unidades al día**.
Se hace UNA promoción de un solo día y se sigue qué pasa ese día y en los 6 días siguientes.

In [13]:
# Se toman de la tabla los 3 coeficientes de la promoción propia de BEVERAGES: hoy, 1 a 3 días atrás y 4 a 6 días atrás
c = consolidada.loc[consolidada["categoria"] == "BEVERAGES"].set_index("variable")["coef"]
b0, b1, b2 = c["onpromotion"], c["onpromotion_l1_3"], c["onpromotion_l4_6"]
base = 100  # unidades que vende normalmente un día sin promoción (ejemplo)

# Cada día tiene su propio "empujón" en escala de coeficiente.
# Los rezagos son promedios de 3 días: con UN solo día en promoción dentro del bloque, cuenta 1/3 del coeficiente.
dias = [
    ("Día 0: el día de la promoción", b0),
    ("Día 1 después", b1 / 3), ("Día 2 después", b1 / 3), ("Día 3 después", b1 / 3),
    ("Día 4 después", b2 / 3), ("Día 5 después", b2 / 3), ("Día 6 después", b2 / 3),
]
semana = pd.DataFrame(dias, columns=["dia", "coeficiente_del_dia"])
semana["ventas_con_promocion"] = base * np.exp(semana["coeficiente_del_dia"])  # e^coef = cuántas veces las ventas normales
semana["unidades_extra"] = semana["ventas_con_promocion"] - base
print(semana.round(1).to_string(index=False))

total_extra = semana["unidades_extra"].sum()
semana_normal = 7 * base
print(f"\nUnidades extra en la semana: {total_extra:.1f}")
print(f"Una semana normal sin promoción vende: {semana_normal} unidades")
print(f"Efecto de la promoción de 1 día en la semana = {total_extra:.1f} / {semana_normal} = {100 * total_extra / semana_normal:+.1f} %")
print(f"(Ese es el número `efecto_acum_1dia` de la tabla: {consolidada.loc[(consolidada['categoria']=='BEVERAGES') & (consolidada['variable']=='efecto_acum_1dia'), 'efecto_pct'].iloc[0]:+.1f} %)")

                          dia  coeficiente_del_dia  ventas_con_promocion  unidades_extra
Día 0: el día de la promoción                  0.5                 159.8            59.8
                Día 1 después                  0.1                 108.3             8.3
                Día 2 después                  0.1                 108.3             8.3
                Día 3 después                  0.1                 108.3             8.3
                Día 4 después                 -0.0                  96.6            -3.4
                Día 5 después                 -0.0                  96.6            -3.4
                Día 6 después                 -0.0                  96.6            -3.4

Unidades extra en la semana: 74.4
Una semana normal sin promoción vende: 700 unidades
Efecto de la promoción de 1 día en la semana = 74.4 / 700 = +10.6 %
(Ese es el número `efecto_acum_1dia` de la tabla: +10.6 %)


**Cómo leer esta tabla.**
- **Día 0:** la promoción hace que se venda ~160 en vez de 100 (+60 unidades).
- **Días 1 a 3 después:** todavía queda algo de "estela": ~108 cada día (+8 unidades por día), porque el modelo ve que 1 de los 3 días anteriores tuvo promoción.
- **Días 4 a 6 después:** la señal es pequeña y no se distingue de cero; se ve como un leve −3 % por día, que en la práctica es ruido.
- **Suma de la semana:** unas +74 unidades sobre 700 que se venderían sin promoción = **+10,6 %**.

Por eso el efecto en la semana (+10,6 %) es mucho menor que el del día (+60 %): **ese +60 % ocurre un solo día de los siete**. El "7" de la cuenta es esos siete días de la semana.

## 9. Comparación final: sin la fórmula del 1 día frente a con la fórmula

- **Sin la fórmula (efecto neto de 7 días seguidos):** suma los tres coeficientes y dice cuánto sube un día cuando el producto lleva **una semana entera** en promoción. Casi nunca ocurre en la realidad.
- **Con la fórmula (efecto de 1 día en la semana):** dice cuánto suma **una promoción de un solo día** a las ventas de la semana. Es la situación normal.

Los dos salen **del mismo modelo y los mismos coeficientes**; la fórmula no cambia la estimación, solo la forma de leerla.

In [14]:
filas = []
for categoria in categorias_objetivo:
    t = consolidada.loc[consolidada["categoria"] == categoria].set_index("variable")
    filas.append({
        "categoria": categoria,
        "efecto_del_dia_%": t.loc["onpromotion", "efecto_pct"],
        "sin_formula_7_dias_seguidos_%": t.loc["NETO_7dias onpromotion", "efecto_pct"],
        "con_formula_1_dia_en_la_semana_%": t.loc["efecto_acum_1dia", "efecto_pct"],
        "IC95_inferior_%": t.loc["efecto_acum_1dia", "efecto_pct_ic95_inf"],
        "IC95_superior_%": t.loc["efecto_acum_1dia", "efecto_pct_ic95_sup"],
        "valor_p_1_dia": t.loc["efecto_acum_1dia", "valor_p"],
    })
comparacion = pd.DataFrame(filas)
comparacion.to_csv(os.path.join(ruta_tablas, "tabla_27_fase1c_comparacion_con_y_sin_formula.csv"), index=False)
print(comparacion.round(2).to_string(index=False))

print("\nLECTURA EN PALABRAS")
for _, f in comparacion.iterrows():
    print(f"- {f['categoria']}: sin la fórmula se diría '+{f['sin_formula_7_dias_seguidos_%']:.0f} %' (promoción 7 días seguidos); "
          f"con la fórmula, una promoción de un día suma +{f['con_formula_1_dia_en_la_semana_%']:.1f} % a la semana "
          f"(rango razonable +{f['IC95_inferior_%']:.1f} % a +{f['IC95_superior_%']:.1f} %).")
print(">>>> Proceso finalizado <<<<")

       categoria  efecto_del_dia_%  sin_formula_7_dias_seguidos_%  con_formula_1_dia_en_la_semana_%  IC95_inferior_%  IC95_superior_%  valor_p_1_dia
       BEVERAGES             59.82                          82.72                             10.62             7.53            13.72            0.0
       GROCERY I             60.43                         105.75                             12.35            10.57            14.14            0.0
        CLEANING             88.94                         132.81                             15.82            12.59            19.05            0.0
           DAIRY             41.22                          58.88                              7.59             6.57             8.61            0.0
    BREAD/BAKERY             45.45                          85.47                             10.04             6.59            13.49            0.0
LIQUOR,WINE,BEER             75.02                          98.72                             12.58       

**Cómo leer esta comparación.**
- **`efecto_del_dia_%`:** lo que sube el producto el mismo día de la promoción (de +41 % en lácteos a +89 % en limpieza).
- **`sin_formula_7_dias_seguidos_%`:** lo que se asocia a mantener la promoción una semana entera. Es el número más grande y el más fácil de malinterpretar como "lo que gana una promoción".
- **`con_formula_1_dia_en_la_semana_%`:** lo que gana la semana con una promoción de un solo día (entre ~8 % y ~16 %). **Es el número recomendado para responder "¿cuánto gano con una promoción?".**
- **`IC95_...`:** el rango donde, con 95 % de confianza, está ese efecto. Si no incluye el cero, el efecto es distinguible de cero (`valor_p_1_dia` menor que 0,05).

**Lo que no cambia:** el modelo, los coeficientes y los valores p son los mismos con o sin la fórmula. Si el profesor prefiere no usarla, se descarta la columna y no se pierde nada del resto.

**Orden de las categorías:** sigue siendo el mismo con las dos lecturas (limpieza responde más, lácteos menos). Cambia el tamaño del número, no la historia.

**Conclusiones y advertencias.**
1. En las 6 categorías sin programa semanal, una promoción de un producto se asocia con ventas claramente mayores ese día, y la semana completa también sube (entre ~8 % y ~16 %). No se observa una caída posterior que lo compense.
2. En varias categorías hay señales de canibalización cercana: cuando otros productos de su misma clase están en promoción, las ventas del producto bajan (ver la lectura automática de arriba).
3. Se habla de **asociación, no de causa**: la cadena decide cuándo promocionar y los controles reducen ese sesgo, pero no lo eliminan.
4. Es una sola tienda, en 2016-2017, sin información de precios ni márgenes: se miden efectos sobre el volumen de ventas, no sobre la rentabilidad.
5. Estos resultados **no incluyen las categorías con programa semanal** (carnes, aves, fiambres, frutas y verduras). Para ellas, la fórmula solo es confiable con los rezagos corregidos, que están pendientes de aprobación del profesor guía.